# 05_COMPLEX_QUERY

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))  # nếu đang ở notebooks/

from typing import Any, Dict, List

from src.agents.planner import decompose_to_subqueries
from src.agents.router import route_query
from src.retrieval.hybrid_search import hybrid_retrieve
from src.agents.synthesizer import synthesize_answer

c:\STUDY\code\medagent-rag-end2end\.med_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _dedupe_contexts(items: List[Dict[str, Any]], max_items: int) -> List[Dict[str, Any]]:
    """Giống `src/graph/nodes.py` — gộp chunk trùng id/text."""
    seen: set[Any] = set()
    out: List[Dict[str, Any]] = []
    for item in items:
        key = item.get("id")
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
        if len(out) >= max_items:
            break
    return out

In [15]:
query = "Men gan tăng nhẹ có tự hết không hay phải uống thuốc bổ gan?"
top_k = 5
retrieval_filter = None

## Step 1: Route

In [16]:
r = route_query(query)
print("route:", r)

route: {'route': 'complex_qa'}


## Step 2: Retrieve

In [17]:
sub_queries = decompose_to_subqueries(query)
print("sub_queries:", sub_queries)

sub_queries: ['Men gan tăng nhẹ có tự hết không?', 'Men gan tăng nhẹ có phải uống thuốc bổ gan không?']


In [19]:
merged: List[Dict[str, Any]] = []
per_sub_k = max(1, top_k)
max_merged = min(40, max(top_k * max(2, len(sub_queries)), top_k))
for sq in sub_queries:
    sq = (sq or "").strip()
    if not sq:
        continue
    part = hybrid_retrieve(query=sq, top_k=per_sub_k, filter=retrieval_filter)
    merged.extend(part)
contexts = _dedupe_contexts(merged, max_items=max_merged)
print("Length of contexts:", len(contexts))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 36232.09it/s]


Length of contexts: 10


### Step 3: Synthesize

In [ ]:
answer = synthesize_answer(query=query, contexts=contexts)
print("answer:", answer)